<a href="https://colab.research.google.com/github/GelsonRibeiroJr/alura-agent-rag/blob/main/1%C2%BA_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Importando Bibliotecas*

In [12]:
%pip install -qU pypdf
%pip install -U langchain
%pip install -U langchain-community
%pip install -U langchain-groq
%pip install langchain-huggingface
%pip install langgraph

Chave de API - **GROQ & NASA**

In [1]:
from google.colab import userdata
import os

# Carrega a chave da Groq
groq_api_key = userdata.get('GROQ_API_KEY')
os.environ['GROQ_API_KEY'] = groq_api_key

# Carrega a chave da NASA
nasa_api_key = userdata.get('NASA_API_KEY')
os.environ['NASA_API_KEY'] = nasa_api_key

# Carrega o token do Hugging Face
hf_token = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = hf_token

print("Chaves de API carregadas com sucesso!")

Chaves de API carregadas com sucesso!


Cloando o repositorio para o ambiente Colab

In [14]:
!git clone https://github.com/GelsonRibeiroJr/alura-agent-rag.git

fatal: destination path 'alura-agent-rag' already exists and is not an empty directory.


In [15]:
import os
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

# Caminho apontando para a pasta baixada do GitHub
path_data = './alura-agent-rag/data'

# Configura o leitor para processar todos os PDFs
loader = DirectoryLoader(
    path_data,
    glob="**/*.pdf",
    loader_cls=PyPDFLoader,
    show_progress=True
)

# Carrega todos os documentos
documents = loader.load()

print(f"Sucesso! Total de páginas processadas: {len(documents)}")

100%|██████████| 29/29 [00:34<00:00,  1.18s/it]

Sucesso! Total de páginas processadas: 400


In [20]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Configura o fatiador de texto
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,   # Tamanho aproximado de cada pedaço (caracteres)
    chunk_overlap=150  # Quantidade de caracteres sobrepostos entre pedaços vizinhos
)

# Aplica a divisão em todas as 400 páginas
chunks = text_splitter.split_documents(documents)

print(f"Base fatiada com sucesso! Total de chunks gerados: {len(chunks)}")
print("\n--- Exemplo do primeiro Chunk gerado ---")
print(chunks[0].page_content[:300]) # Mostra os primeiros 300 caracteres do 1º pedaço

Base fatiada com sucesso! Total de chunks gerados: 890

--- Exemplo do primeiro Chunk gerado ---
IAC-24.B3.1.11                           Page 1 of 8 
IAC-24,B3,1,11,x83343 
 
NASA’s Human Landing System Program: Progress Toward Artemis III and Beyond  
 
Dr. Lisa Watson-Morgana, Dr. Kent Chojnacki*, Rene Ortegac, Dr. Thomas K. Percyd, John Crislere 
 
aNational Aeronautics and Space Administra


In [21]:
# 1. Instala a biblioteca de embeddings da HuggingFace e o banco vetorial FAISS
%pip install -q sentence-transformers faiss-cpu

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("Carregando o modelo de Embeddings...")
# Usamos um modelo multilíngue super eficiente e leve
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

print("Gerando os vetores no banco de dados (FAISS)... Isso leva cerca de 30 a 60 segundos.")
vectorstore = FAISS.from_documents(documents=chunks, embedding=embeddings)

print("✅ Banco Vetorial criado com sucesso! Todos os 890 chunks foram indexados.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 59.3 MB/s eta 0:00:00
Carregando o modelo de Embeddings...


/tmp/ipykernel_2956/1648799614.py:9: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Gerando os vetores no banco de dados (FAISS)... Isso leva cerca de 30 a 60 segundos.
✅ Banco Vetorial criado com sucesso! Todos os 890 chunks foram indexados.


In [22]:
# Teste de busca por similaridade semântica
pergunta = "Qual é o objetivo do programa Human Landing System no projeto Artemis?"

# Busca os 3 pedaços de texto mais parecidos no banco
resultados = vectorstore.similarity_search(pergunta, k=3)

print("--- Trechos mais relevantes encontrados pelo Banco Vetorial ---")
for i, doc in enumerate(resultados):
    print(f"\nResultado {i+1}:")
    print(doc.page_content[:400]) # Exibe os 400 primeiros caracteres do resultado

--- Trechos mais relevantes encontrados pelo Banco Vetorial ---

Resultado 1:
AAS 23-057
AN OVERVIEW OF THE ARTEMIS I NAVIGATION PERFORMANCE
Greg Holt*, Chris D’Souza †, and Michael Wasinger ‡
The goal of NASA’s Artemis Program is to explore the Moon and beyond. The
Artemis I Mission which flew in late 2022 was the uncrewed test flight whose goal
was to exercise the entire navigation system in an extended duration flight and
evaluate its performance over the entire mission,

Resultado 2:
Artemis program will land the first woman and next man on the surface of the Moon and 
establish, together with international and commercial partners, the sustainable human exploration 
of the solar system; 
 
CONSIDERING the necessity of greater coordination and cooperation between and among 
established and emerging actors in space; 
 
RECOGNIZING the global benefits of space exploration and com

Resultado 3:
11
Chapter 1: Setting Humanity on a Sustainable Course  
to the Moon
The Artemis program bui

Importando **ChatGroq**

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Inicializa o modelo LLM da Groq
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.2, # Baixa temperatura para manter fidelidade aos documentos
    api_key=os.environ.get("GROQ_API_KEY")
)

# Teste rápido direto do modelo (sem RAG)
resposta_teste = llm.invoke("Diga 'Modelo Groq conectado com sucesso!' em português.")
print(resposta_teste.content)

Configurando a Cadeia RAG

In [19]:
# 1. Configura o Prompt do Sistema (Instruindo a resposta em Português)
template = """Você é um especialista em exploração espacial assistente do Programa Artemis da NASA.
Responda à pergunta do usuário utilizando APENAS o contexto fornecido abaixo.
Se a informação não estiver no contexto, diga honestamente que não encontrou a informação nos documentos.
Se os documentos estiverem em inglês, faça a tradução e responda de forma clara, precisa e em Português do Brasil.

Contexto dos documentos da NASA:
{context}

Pergunta do usuário:
{question}

Resposta em Português do Brasil:"""

prompt = ChatPromptTemplate.from_template(template)

# 2. Transforma o banco FAISS em um recuperador (Retriever)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# Helper para formatar os documentos encontrados em um texto único
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 3. Monta a cadeia RAG
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ Pipeline RAG configurado com sucesso!")

Teste do RAG em Funcionamento

In [ ]:
# Faz uma pergunta técnica sobre a base da NASA
pergunta = "Qual foi a função da missão Artemis I e quando ela ocorreu?"

print(f"Pergunta: {pergunta}\n")
print("Buscando nos documentos e gerando resposta...\n")

resposta_rag = rag_chain.invoke(pergunta)

print("=== RESPOSTA DO RAG DA NASA ===")
print(resposta_rag)